# Chapter 15: The Black–Scholes–Merton Model

## Concise Summary

This chapter frames Black-Scholes-Merton as both a pricing engine and a bundle of strong assumptions. The model relies on lognormal stock dynamics, constant volatility and rates, continuous trading, no transaction costs, no arbitrage, and perfect hedging through continuous rebalancing. Its key insight is that option value is independent of the stock's expected return: risk-neutral valuation replaces the real-world drift with the risk-free rate while volatility, time, moneyness, dividends, and discounting drive price. That makes BSM useful for benchmarking, but dangerous if its assumptions are forgotten.

The daily takeaway is to validate BSM outputs through assumptions, sensitivities, and market calibration, not formula correctness alone. Historical volatility estimates depend on sampling choices, trading-day conventions, and regime stability, while implied volatility embeds forward-looking market information and often exposes smiles, skews, and stress expectations that flat-vol BSM cannot explain. Dividend treatment, early-exercise logic, and dilution adjustments can materially change valuation. For governance, document the risk-neutral measure, volatility source, day-count convention, dividend handling, and numerical solver for implied volatility. Stress tests should cover volatility shocks, hedging error from discrete rebalancing, liquidity gaps, and parameter instability.

> Summary of key definitions, theorems, and applications from Hull's *Options, Futures, and Other Derivatives*

**Legend:**
- 🟡 **Yellow** = Definition / Theorem / Property
- 🟢 **Green** = Comment / Application

---

## 15.1 Lognormal Property of Stock Prices

### 🟡 Property: Distribution of Stock Prices
Under GBM, the stock price change satisfies:

$$\frac{\Delta S}{S} \sim \phi(\mu\,\Delta t,\; \sigma^2\,\Delta t)$$

Applying Itô's lemma (from Chapter 14):

$$\ln S_T - \ln S_0 \sim \phi\left[\left(\mu - \frac{\sigma^2}{2}\right)T,\; \sigma^2 T\right]$$

$$\ln S_T \sim \phi\left[\ln S_0 + \left(\mu - \frac{\sigma^2}{2}\right)T,\; \sigma^2 T\right]$$

Since $\ln S_T$ is normally distributed, $S_T$ is **lognormally distributed**.

### 🟡 Properties of the Lognormal Distribution

$$E(S_T) = S_0 e^{\mu T}$$

$$\text{Var}(S_T) = S_0^2 e^{2\mu T}(e^{\sigma^2 T} - 1)$$

- The lognormal distribution is **skewed** (mean ≠ median ≠ mode)
- $S_T$ can take any value between 0 and $\infty$

### 🟢 Application: Confidence Interval
For $S_0 = 40$, $\mu = 0.16$, $\sigma = 0.20$, $T = 0.5$:
- $\ln S_T \sim \mathcal{N}(3.759, 0.02)$, SD $= 0.141$
- 95% CI for $S_T$: $(32.55, 56.56)$

## 15.2 The Distribution of the Rate of Return

### 🟡 Property: Continuously Compounded Return
The continuously compounded return $x$ over period $T$ is:

$$x = \frac{1}{T}\ln\frac{S_T}{S_0}$$

$$\boxed{x \sim \phi\left(\mu - \frac{\sigma^2}{2},\; \frac{\sigma^2}{T}\right)}$$

- Mean: $\mu - \sigma^2/2$ (not $\mu$!)
- SD: $\sigma / \sqrt{T}$ — decreases with horizon (we're more certain about the *average* return over long periods)

### 🟢 Comment: Why $E(x) = \mu - \sigma^2/2 \neq \mu$
- $\mu$ is the **arithmetic** average of short-period returns
- The **geometric** average (realized compound return) is $\mu - \sigma^2/2$
- $\ln[E(S_T)] \neq E[\ln(S_T)]$ because $\ln$ is nonlinear (Jensen's inequality)

### 🟢 Application: Mutual Fund Returns (Business Snapshot 15.1)
Annual returns: 15%, 20%, 30%, −20%, 25% → arithmetic mean = 14%, but geometric mean = 12.4%. The geometric mean is always less than the arithmetic mean. This is why fund reporting can be misleading.

## 15.3 The Expected Return

### 🟢 Comment
- $\mu$ depends on the stock's risk and the level of interest rates
- **Crucially**: the value of a stock option, expressed in terms of the stock price, **does not depend on $\mu$ at all**
- The term "expected return" is ambiguous: it can refer to $\mu$ (arithmetic) or $\mu - \sigma^2/2$ (geometric/continuously compounded)

## 15.4 Volatility

### 🟡 Definition: Volatility
The **volatility** $\sigma$ is the standard deviation of the continuously compounded return provided by the stock in 1 year.

- $\sigma\sqrt{\Delta t}$ ≈ standard deviation of the percentage change in price over $\Delta t$
- Typical range: 15% to 60% per annum
- Uncertainty about future price grows as $\approx \sigma\sqrt{T}$

### 🟡 Estimating Volatility from Historical Data
Given $n+1$ price observations $S_0, S_1, \ldots, S_n$ at intervals of $\tau$ years:

$$u_i = \ln\left(\frac{S_i}{S_{i-1}}\right)$$

$$s = \sqrt{\frac{1}{n-1}\sum_{i=1}^n (u_i - \bar{u})^2}$$

$$\hat{\sigma} = \frac{s}{\sqrt{\tau}}, \quad \text{SE}(\hat{\sigma}) \approx \frac{\hat{\sigma}}{\sqrt{2n}}$$

### 🟢 Application: Trading Days vs. Calendar Days
- Research shows volatility is **much higher** when the exchange is open than when closed
- Practitioners use **trading days** (not calendar days) when estimating volatility and measuring option life
- Convention: **252 trading days per year**
- Option maturity: $T = \text{trading days to maturity} / 252$

### 🟢 Comment: What Causes Volatility? (Business Snapshot 15.2)
- Friday-to-Monday variance is only ~1.1–1.2× the daily variance (not 3×)
- Conclusion: volatility is largely **caused by trading itself**, not just by news arrival

## 15.5 The Idea Underlying the Black–Scholes–Merton Differential Equation

### 🟢 Core Intuition
The approach mirrors the binomial-model no-arbitrage argument:

1. **Construct a riskless portfolio** of the derivative and the stock
2. Stock price and derivative price share the **same source of uncertainty** ($dz$)
3. Gains/losses on one position **offset** the other over a short time interval
4. The riskless portfolio must earn the **risk-free rate** $r$

> **Key difference from the binomial model:** the portfolio is riskless only for an **instantaneously short** period and must be **continuously rebalanced**.

**Example:** If $\Delta c = 0.4\,\Delta S$, then a riskless portfolio is: long 40 shares + short 100 calls.

## 15.6 Derivation of the Black–Scholes–Merton Differential Equation

### 🟡 Assumptions
1. Stock follows GBM with constant $\mu$ and $\sigma$
2. Short selling with full use of proceeds is permitted
3. No transaction costs or taxes; all securities perfectly divisible
4. No dividends during the life of the derivative
5. No riskless arbitrage opportunities
6. Continuous trading
7. Risk-free rate $r$ is constant for all maturities

### 🟡 The BSM Differential Equation
From GBM and Itô's lemma, construct a portfolio $\Pi = -f + \frac{\partial f}{\partial S}S$ that eliminates $dz$:

$$\Delta\Pi = \left(-\frac{\partial f}{\partial t} - \frac{1}{2}\frac{\partial^2 f}{\partial S^2}\sigma^2 S^2\right)\Delta t$$

Setting $\Delta\Pi = r\Pi\,\Delta t$ (no-arbitrage):

$$\boxed{\frac{\partial f}{\partial t} + rS\frac{\partial f}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 f}{\partial S^2} = rf}$$

### 🟢 Key Observations
- The equation **does not involve $\mu$** — it dropped out! Only $S$, $t$, $\sigma$, and $r$ appear
- Any function $f(S, t)$ satisfying this PDE is a valid (no-arbitrage) derivative price
- The specific derivative is determined by **boundary conditions**:
  - European call: $f = \max(S - K, 0)$ at $t = T$
  - European put: $f = \max(K - S, 0)$ at $t = T$

### 🟢 Application: Forward Contract Verification
For $f = S - Ke^{-r(T-t)}$: $\;f_t = -rKe^{-r(T-t)}$, $f_S = 1$, $f_{SS} = 0$. Substituting into the PDE gives $rf$. ✓

### 🟢 Application: Perpetual Derivative
A derivative paying $Q$ when $S$ first hits $H$ (with $S > H$):

$$f = Q\left(\frac{S}{H}\right)^{-2r/\sigma^2}$$

## 15.7 Risk-Neutral Valuation

### 🟡 Theorem: Risk-Neutral Valuation
Since the BSM PDE is **independent of risk preferences** (no $\mu$), we can assume investors are risk-neutral:

**Procedure:**
1. Assume the expected return from the stock is $r$ (i.e., set $\mu = r$)
2. Calculate the **expected payoff** of the derivative
3. **Discount** at the risk-free rate $r$

$$f = e^{-rT}\,\hat{E}[\text{payoff}]$$

where $\hat{E}$ denotes expectation in the risk-neutral world.

### 🟢 Comment
- Risk-neutral valuation is an **artificial device** — solutions are valid in **all worlds**, not just risk-neutral ones
- When moving to a risk-averse world, the expected payoff changes and the discount rate changes, but these **exactly offset**

### 🟢 Application: Forward Contract
$$f = e^{-rT}\hat{E}(S_T - K) = e^{-rT}[S_0 e^{rT} - K] = S_0 - Ke^{-rT}$$
Matches the known result. ✓

## 15.8 Black–Scholes–Merton Pricing Formulas

### 🟡 Theorem: BSM Formulas

**European Call:**
$$\boxed{c = S_0\,N(d_1) - K e^{-rT}\,N(d_2)}$$

**European Put:**
$$\boxed{p = K e^{-rT}\,N(-d_2) - S_0\,N(-d_1)}$$

where:

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$$

$$d_2 = \frac{\ln(S_0/K) + (r - \sigma^2/2)T}{\sigma\sqrt{T}} = d_1 - \sigma\sqrt{T}$$

$N(x)$ is the cumulative standard normal distribution function.

### 🟡 Interpretation
| Term | Meaning |
|---|---|
| $N(d_2)$ | Probability the call is exercised (risk-neutral) |
| $S_0 N(d_1) e^{rT}$ | Expected stock price at $T$ when $S_T < K$ is counted as zero |
| $e^{-rT}$ | Present value factor |

Alternative reading: $c = e^{-rT} N(d_2)[S_0 e^{rT} N(d_1)/N(d_2) - K]$

### 🟡 Properties (Limiting Behavior)
- As $S_0 \to \infty$: $c \to S_0 - Ke^{-rT}$ (like a forward), $p \to 0$
- As $\sigma \to 0$: $c \to \max(S_0 - Ke^{-rT}, 0)$, $p \to \max(Ke^{-rT} - S_0, 0)$

### 🟢 Application: Numerical Example
$S_0 = 42$, $K = 40$, $r = 0.10$, $\sigma = 0.20$, $T = 0.5$:
- $d_1 = 0.7693$, $d_2 = 0.6278$
- $c = 42 \times 0.7791 - 38.049 \times 0.7349 = 4.76$
- $p = 38.049 \times 0.2651 - 42 \times 0.2209 = 0.81$

### 🟢 Note on American Options
- American **call** on non-dividend-paying stock: equation (15.20) applies (early exercise never optimal)
- American **put**: no closed-form solution; requires numerical methods (Chapter 21)

## 15.10 Warrants and Employee Stock Options

### 🟡 Property: Dilution Effect
Each warrant on a company with $N$ shares outstanding and $M$ new warrants is worth:

$$\text{Warrant value} = \frac{N}{N + M} \times \text{(BSM call value)}$$

### 🟢 Comment on Dilution
- If markets are efficient, potential dilution is **already reflected** in the stock price at announcement
- The payoff to a warrant holder upon exercise is $\frac{N}{N+M}(S_T - K)$, not $(S_T - K)$

### 🟢 Application
Company: 1M shares at \$40, issuing 200K warrants ($K = 60$, $T = 5$, $r = 3\%$, $\sigma = 30\%$). BSM call = \$7.04.
- Warrant value: $\frac{1{,}000{,}000}{1{,}200{,}000} \times 7.04 = \$5.87$
- Total cost: $200{,}000 \times 5.87 = \$1.17M$
- Expected stock price decline: \$1.17

## 15.11 Implied Volatilities

### 🟡 Definition: Implied Volatility
The value of $\sigma$ that, when substituted into the BSM formula, gives the **observed market price** of the option.

- Cannot be solved analytically — found by **iterative search** (e.g., bisection, Newton–Raphson)
- **Forward-looking** (vs. historical volatility which is backward-looking)
- Traders often quote options by their implied volatility rather than price

### 🟢 Application: The VIX Index
- **VIX** = implied volatility of 30-day options on the S&P 500 (the "fear factor")
- VIX of 15 → implied vol ≈ 15%
- Futures on VIX since 2004; options on VIX since 2006
- Reached ~80 in Oct/Nov 2008 (Lehman); spiked again in 2020 (COVID-19)
- A VIX trade is a **bet on volatility** itself, not on the S&P 500 level

## 15.12 Dividends

### 🟡 Property: European Options on Dividend-Paying Stocks
Use BSM with the stock price **reduced** by the present value of dividends during the option's life:

$$S_0^* = S_0 - \sum_i D_i\,e^{-r t_i}$$

where $D_i$ is the dividend at ex-dividend date $t_i$. Plug $S_0^*$ into the standard BSM formulas.

### 🟢 Application
$S_0 = 40$, $K = 40$, $\sigma = 0.30$, $r = 0.09$, $T = 0.5$; dividends of \$0.50 at months 2 and 5.

$S_0^* = 40 - 0.50e^{-0.09 \times 2/12} - 0.50e^{-0.09 \times 5/12} = 39.0258$

Result: $c = \$3.67$

### 🟡 Property: Early Exercise of American Calls
- Early exercise can only be optimal **just before an ex-dividend date**
- It is **not optimal** to exercise before date $t_i$ if:
$$D_i \leq K[1 - e^{-r(t_{i+1} - t_i)}]$$
- If this holds for all dividend dates, the American call = European call

### 🟢 Application: Black's Approximation
Compute two European option prices:
1. One maturing at $T$
2. One maturing just before the final ex-dividend date $t_n$

Set the American call price = $\max$ of the two. This is approximate because it assumes the exercise decision is made at time 0.

## Summary

| Topic | Key Result |
|---|---|
| Stock price distribution | $\ln S_T \sim \mathcal{N}\left(\ln S_0 + (\mu - \sigma^2/2)T,\; \sigma^2 T\right)$ |
| Realized return distribution | $x \sim \mathcal{N}(\mu - \sigma^2/2,\; \sigma^2/T)$ |
| BSM PDE | $f_t + rSf_S + \frac{1}{2}\sigma^2 S^2 f_{SS} = rf$ |
| Risk-neutral valuation | $f = e^{-rT}\hat{E}[\text{payoff}]$ with $\mu \to r$ |
| European call | $c = S_0 N(d_1) - Ke^{-rT}N(d_2)$ |
| European put | $p = Ke^{-rT}N(-d_2) - S_0 N(-d_1)$ |
| Implied volatility | $\sigma$ that equates BSM price to market price |
| Dividends | Replace $S_0$ with $S_0 - PV(\text{dividends})$ |

### Key Takeaways
1. The BSM equation is **independent of $\mu$** — enabling risk-neutral valuation
2. **Risk-neutral valuation**: set $\mu = r$, compute expected payoff, discount at $r$
3. **Implied volatility** is the market's forward-looking estimate of $\sigma$
4. Volatility estimation uses **trading days**, not calendar days (252/year)
5. For dividend-paying stocks, subtract PV of dividends from $S_0$
6. American calls may be exercised early only just **before ex-dividend dates**

## Appendix: Proof of BSM Formula via Risk-Neutral Valuation

### 🟡 Key Result (Lognormal Expectation)
If $V$ is lognormally distributed with $\text{SD}(\ln V) = w$, then:

$$E[\max(V - K, 0)] = E(V)\,N(d_1) - K\,N(d_2)$$

where:
$$d_1 = \frac{\ln[E(V)/K] + w^2/2}{w}, \quad d_2 = \frac{\ln[E(V)/K] - w^2/2}{w}$$

### 🟢 Deriving BSM
Apply the key result with:
- $V = S_T$ (lognormal in risk-neutral world)
- $\hat{E}(S_T) = S_0 e^{rT}$
- $w = \sigma\sqrt{T}$

$$c = e^{-rT}\hat{E}[\max(S_T - K, 0)] = e^{-rT}[S_0 e^{rT} N(d_1) - KN(d_2)] = S_0 N(d_1) - Ke^{-rT}N(d_2)$$

which is exactly the Black–Scholes–Merton call formula. ∎